# Survival Analysis — PBTA_RNA Clinical Deep Dive

**Phase 1: Outcome Analysis (OS & EFS)**

This notebook tests associations between clinical variables (AGE, TUMOR_FRACTION, TUMOR_PLOIDY, SEX, CANCER_PREDISPOSITIONS) and patient outcomes using:
- **Binary (STATUS):** Mann-Whitney U / Chi-squared
- **Time-to-event (KM + log-rank):** Dichotomized variables, Kaplan-Meier curves

Per-cancer-group analyses for groups with n ≥ 20 samples.

For shared methodology, see `context/clinical_deep_dive_general.md`.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency, spearmanr
from scipy.stats import fisher_exact, ks_2samp, shapiro
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sp
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/home/alon/menow_home_ass/PBTA_RNA"
PATIENT_FILE = f"{DATA_DIR}/data_clinical_patient_attributes.txt"
SAMPLE_FILE = f"{DATA_DIR}/data_clinical_sample_attributes.txt"


In [2]:
def read_patients():
    return pd.read_csv(PATIENT_FILE, sep="\t", header=4,
                       dtype={"AGE": float, "AGE_IN_DAYS": float,
                              "OS_MONTHS": float, "EFS_MONTHS": float})

def read_samples():
    return pd.read_csv(SAMPLE_FILE, sep="\t", header=4)

def clean_os(df):
    df = df.copy()
    df["OS_STATUS"] = df["OS_STATUS"].str.strip()
    df["os_label"] = df["OS_STATUS"].str.replace(r"^\d+:", "", regex=True)
    df["os_event"] = df["OS_STATUS"].apply(
        lambda x: 1 if pd.notna(x) and x.startswith("1:") else (0 if pd.notna(x) and x.startswith("0:") else np.nan))
    return df

def clean_efs(df):
    df = df.copy()
    df["EFS_STATUS"] = df["EFS_STATUS"].str.strip()
    df["efs_detail"] = df["EFS_STATUS"].str.replace(r"^\d+:", "", regex=True)
    df["efs_event"] = df["EFS_STATUS"].apply(
        lambda x: 0 if pd.notna(x) and x == "0:No Event"
        else (1 if pd.notna(x) and x != "1:NA" else np.nan))
    return df

def clean_race_eth(df):
    df = df.copy()
    df["RACE"] = df["RACE"].fillna("Unknown").replace({"Not Reported":"Unknown","Reported Unknown":"Unknown"})
    df["ETHNICITY"] = df["ETHNICITY"].fillna("Unknown").replace({"Not Reported":"Unknown","Reported Unknown":"Unknown"})
    return df

def clean_pred(df):
    df = df.copy()
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].fillna("Unknown")
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].replace("Not Reported","Unknown")
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].replace("None documented","No predisposition")
    return df

def clean_subtype(df):
    df = df.copy()
    df["MOLECULAR_SUBTYPE"] = df["MOLECULAR_SUBTYPE"].fillna("Unclassified")
    df["MOLECULAR_SUBTYPE"] = df["MOLECULAR_SUBTYPE"].replace("To be classified","Unclassified")
    return df

def clean_tf_tp(df):
    df = df.copy()
    df["TF_group"] = np.where(df["TUMOR_FRACTION"].isna(), "Unknown", "Measured")
    df["TP_group"] = np.where(df["TUMOR_PLOIDY"].isna(), "Unknown", "Measured")
    return df

def cliffs_delta(x, y):
    n_x, n_y = len(x), len(y)
    if n_x == 0 or n_y == 0: return 0
    more = sum(1 for xi in x for yi in y if xi > yi) / (n_x * n_y)
    less = sum(1 for xi in x for yi in y if xi < yi) / (n_x * n_y)
    return more - less

def epsilon_sq(H, k, n):
    return (H - k + 1) / (n - k) if n > k else 0

def cramers_v(ct):
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    phi2 = chi2 / n
    r, k = ct.shape
    return np.sqrt(phi2 / min(k - 1, r - 1))

def kaplan_meier(times, events):
    d = pd.DataFrame({"t": times, "e": events}).dropna().sort_values("t")
    surv = 1.0
    n = len(d)
    res = []
    for t, grp in d.groupby("t", sort=False):
        ne = int(grp["e"].sum())
        if ne > 0:
            surv *= (1 - ne / n)
        res.append({"t": t, "s": surv, "n": n, "ne": ne})
        n -= len(grp)
    return pd.DataFrame(res)

def add_km(fig, km, label, color, row=None, col=None):
    fig.add_trace(go.Scatter(
        x=km["t"], y=km["s"], mode="lines", name=label,
        line=dict(color=color, width=2, shape="hv"),
        legendgroup=label,
        hovertemplate=f"Time: %{{x}}<br>Survival: %{{y:.3f}}<extra>{label}</extra>"),
        row=row, col=col)
    return fig

def logrank2(t1, e1, t2, e2):
    from scipy.stats import chi2
    all_t = sorted(set(pd.concat([pd.Series(t1.dropna()), pd.Series(t2.dropna())]).dropna()))
    if len(all_t) < 2: return 1.0
    o1e = 0; v = 0
    d1 = pd.DataFrame({"t": t1, "e": e1}).dropna()
    d2 = pd.DataFrame({"t": t2, "e": e2}).dropna()
    for t in all_t:
        r1 = (d1["t"] >= t).sum(); r2 = (d2["t"] >= t).sum(); nr = r1 + r2
        if nr == 0: continue
        o1 = int(((d1["t"] == t) & (d1["e"] == 1)).sum())
        o2 = int(((d2["t"] == t) & (d2["e"] == 1)).sum())
        ot = o1 + o2
        if ot == 0: continue
        e1 = ot * r1 / nr; o1e += (o1 - e1)
        if nr > 1: v += ot * (r1 / nr) * (r2 / nr) * (nr - ot) / (nr - 1)
    if v <= 0: return 1.0
    return 1 - chi2.cdf(o1e ** 2 / v, 1)

def logrank_multi(groups):
    from scipy.stats import chi2
    ng = len(groups)
    if ng < 2: return 1.0
    all_t = sorted(set(pd.concat([pd.Series(g[0].dropna()) for g in groups]).dropna()))
    if len(all_t) < 2: return 1.0
    O = np.zeros(ng); E = np.zeros(ng); V = np.zeros((ng, ng))
    for t in all_t:
        ar = np.array([(g[0] >= t).sum() for g in groups]); nr = ar.sum()
        if nr == 0: continue
        ev = np.array([((g[0] == t) & (g[1] == 1)).sum() for g in groups]); ot = ev.sum()
        if ot == 0: continue
        O += ev; E += ot * ar / nr
        if nr > 1:
            for i in range(ng):
                for j in range(ng):
                    if i == j: V[i, j] += ot * ar[i] / nr * (1 - ar[i] / nr) * (nr - ot) / (nr - 1)
                    else: V[i, j] -= ot * ar[i] / nr * ar[j] / nr * (nr - ot) / (nr - 1)
    try: return 1 - chi2.cdf((O - E) @ np.linalg.pinv(V) @ (O - E), ng - 1)
    except: return 1.0

def per_group_km(data, group_col, time_col, event_col, split_col, split_fn, min_n=20):
    results = []
    km_curves = {}
    for group in data[group_col].unique():
        gdata = data[data[group_col] == group].dropna(subset=[time_col, event_col, split_col])
        if len(gdata) < min_n:
            continue
        gdata = gdata.copy()
        gdata['split_group'] = split_fn(gdata, split_col)
        if gdata['split_group'].nunique() < 2:
            continue
        grps = [(gdata[gdata['split_group'] == v][time_col], gdata[gdata['split_group'] == v][event_col])
                for v in gdata['split_group'].unique()]
        if len(grps) < 2:
            continue
        p = logrank_multi(grps)
        n_events = int(gdata[event_col].sum())
        n_total = len(gdata)
        results.append({'group': group, 'n': n_total, 'n_events': n_events, 'p_value': p})
        curves = {}
        for v in gdata['split_group'].unique():
            sub = gdata[gdata['split_group'] == v]
            curves[v] = kaplan_meier(sub[time_col], sub[event_col])
        km_curves[group] = curves
    return results, km_curves

print("Setup complete.")


Setup complete.


In [3]:
patients = read_patients()
samples = read_samples()
patients = clean_os(patients)
patients = clean_efs(patients)
patients = clean_race_eth(patients)
patients = clean_pred(patients)
samples = clean_subtype(samples)
samples = clean_tf_tp(samples)
merged = samples.merge(patients, on='PATIENT_ID', how='left')
print(f"Merged data: {merged.shape[0]} rows, {merged['PATIENT_ID'].nunique()} patients, {merged['SAMPLE_ID'].nunique()} samples")
sample_pids = set(samples['PATIENT_ID'].unique())
patient_pids = set(patients['PATIENT_ID'].unique())
orphan_samples = sample_pids - patient_pids
orphan_patients = patient_pids - sample_pids
print(f"Orphan samples (no patient record): {len(orphan_samples)}")
print(f"Orphan patients (no sample record): {len(orphan_patients)}")


Merged data: 4312 rows, 2870 patients, 4312 samples
Orphan samples (no patient record): 0
Orphan patients (no sample record): 0


In [4]:
MIN_SAMPLES = 20
os_data = merged.dropna(subset=['OS_MONTHS', 'os_event'])
per_group_n = os_data.groupby('CANCER_GROUP').size()
target_groups = per_group_n[per_group_n >= MIN_SAMPLES].index.tolist()
print(f"Groups with n \u2265 {MIN_SAMPLES} for OS analysis: {len(target_groups)}")
for g in target_groups:
    print(f"  {g}: {per_group_n[g]} samples")
excluded = per_group_n[per_group_n < MIN_SAMPLES]
if len(excluded) > 0:
    print(f"Excluded groups (n < {MIN_SAMPLES}):")
    for g in excluded.index:
        print(f"  {g}: {per_group_n[g]} samples")

results_rows = []
def record_result(phase, comparison, test, group, n, n_events, statistic, p_value, effect_size, basic_ref):
    results_rows.append({
        'Phase': phase, 'Comparison': comparison, 'Test': test,
        'Group': group, 'N': n, 'N_events': n_events,
        'Statistic': statistic, 'p_value': round(p_value, 4),
        'Significant': '\U0001f7e2\U0001f7e2\U0001f7e2 p<0.001' if p_value < 0.001 else ('\U0001f7e2\U0001f7e2 p<0.01' if p_value < 0.01 else ('\U0001f7e2 p<0.05' if p_value < 0.05 else '\u274c NS')),
        'Effect_Size': effect_size, 'Basic_Ref': basic_ref
    })

print("Setup complete. Results table initialized.")


Groups with n ≥ 20 for OS analysis: 22
  Adamantinomatous Craniopharyngioma: 98 samples
  Atypical Teratoid Rhabdoid Tumor: 141 samples
  CNS Embryonal tumor: 24 samples
  Chordoma: 26 samples
  Choroid plexus tumor: 89 samples
  Diffuse hemispheric glioma: 26 samples
  Diffuse midline glioma: 368 samples
  Dysembryoplastic neuroepithelial tumor: 47 samples
  Embryonal tumor with multilayer rosettes: 22 samples
  Ependymoma: 296 samples
  Ewing sarcoma: 34 samples
  Ganglioglioma: 123 samples
  Glial-neuronal tumor NOS: 47 samples
  High-grade glioma: 406 samples
  Low-grade glioma: 712 samples
  Medulloblastoma: 383 samples
  Meningioma: 87 samples
  Neuroblastoma: 22 samples
  Neurofibroma/Plexiform: 39 samples
  Pineoblastoma: 27 samples
  Sarcoma: 45 samples
  Schwannoma: 46 samples
Excluded groups (n < 20):
  Astroblastoma: 1 samples
  Astrocytoma: 3 samples
  CNS Burkitt's lymphoma: 1 samples
  CNS neuroblastoma: 4 samples
  Cavernoma: 5 samples
  Central neurocytoma: 4 samples
 

In [5]:
print("Results summary table (will be populated as analyses run):")
print(f"{'Phase':6s} {'Comparison':40s} {'Test':20s} {'Group':30s} {'N':6s} {'p':8s} {'Significant':15s}")
print("-"*130)


Results summary table (will be populated as analyses run):
Phase  Comparison                               Test                 Group                          N      p        Significant    
----------------------------------------------------------------------------------------------------------------------------------


## Section 1A: AGE × OS/EFS

**Rationale:** Age at diagnosis is a known prognostic factor in many pediatric brain tumors. Younger patients may have different tumor biology and different outcomes.

**Basic notebook reference:** Step 14 shows AGE × CANCER_GROUP distribution. This extends to outcome.


In [6]:
# Validation: AGE × OS/EFS sample sizes
age_os = merged.dropna(subset=['AGE', 'OS_MONTHS', 'os_event'])
age_efs = merged.dropna(subset=['AGE', 'EFS_MONTHS', 'efs_event'])
print(f"AGE × OS: {len(age_os)} samples with complete data")
print(f"  Median age: {age_os['AGE'].median():.1f} years (range: {age_os['AGE'].min():.0f}-{age_os['AGE'].max():.0f})")
print(f"  Deceased: {int(age_os['os_event'].sum())} / {len(age_os)}")
print(f"AGE × EFS: {len(age_efs)} samples with complete data")
print(f"  Median age: {age_efs['AGE'].median():.1f} years")
print(f"  Events: {int(age_efs['efs_event'].sum())} / {len(age_efs)}")
print(f"\nAGE missing: {merged['AGE'].isna().sum()} / {len(merged)} ({merged['AGE'].isna().mean()*100:.1f}%)")


AGE × OS: 3475 samples with complete data
  Median age: 7.0 years (range: 0-60)
  Deceased: 1352 / 3475
AGE × EFS: 3358 samples with complete data
  Median age: 7.0 years
  Events: 2165 / 3358

AGE missing: 81 / 4312 (1.9%)


In [7]:
# P1A.1: AGE × OS_STATUS (global) -- Boxplot + Mann-Whitney
import re
df = merged.dropna(subset=['AGE', 'OS_STATUS']).copy()
df['os_label'] = df['OS_STATUS'].str.replace(r'^\d+:', '', regex=True)
living = df[df['os_label'] == 'LIVING']['AGE']
deceased = df[df['os_label'] == 'DECEASED']['AGE']
u_stat, p_val = mannwhitneyu(living, deceased, alternative='two-sided')
d = cliffs_delta(living.values, deceased.values)

fig = go.Figure()
fig.add_trace(go.Box(y=living, name='LIVING', boxmean='sd', 
                     marker_color='#2ecc71', line_color='#27ae60'))
fig.add_trace(go.Box(y=deceased, name='DECEASED', boxmean='sd',
                     marker_color='#e74c3c', line_color='#c0392b'))
fig.update_layout(
    title=f'AGE by OS_STATUS (global) — Mann-Whitney U={u_stat:.0f}, p={p_val:.4f}, d={d:.3f}',
    yaxis_title='Age at diagnosis (years)', height=450,
    template='plotly_white')
fig.show()

record_result('1A', 'AGE × OS_STATUS (global)', 'Mann-Whitney', 'global',
              len(df), int(df['os_label'].eq('DECEASED').sum()),
              f'U={u_stat:.0f}', p_val, f'd={d:.3f} ({"large" if abs(d)>=0.474 else "medium" if abs(d)>=0.33 else "small" if abs(d)>=0.147 else "negligible"})',
              'Step 14')


In [8]:
# P1A.2: AGE × EFS_STATUS (global) -- Boxplot + Mann-Whitney
df = merged.dropna(subset=['AGE', 'efs_event']).copy()
no_event = df[df['efs_event'] == 0]['AGE']
event = df[df['efs_event'] == 1]['AGE']
u_stat, p_val = mannwhitneyu(no_event, event, alternative='two-sided')
d = cliffs_delta(no_event.values, event.values)

fig = go.Figure()
fig.add_trace(go.Box(y=no_event, name='No Event', boxmean='sd',
                     marker_color='#3498db', line_color='#2980b9'))
fig.add_trace(go.Box(y=event, name='Event', boxmean='sd',
                     marker_color='#e67e22', line_color='#d35400'))
fig.update_layout(
    title=f'AGE by EFS_STATUS (global) — Mann-Whitney U={u_stat:.0f}, p={p_val:.4f}, d={d:.3f}',
    yaxis_title='Age at diagnosis (years)', height=450,
    template='plotly_white')
fig.show()

record_result('1A', 'AGE × EFS_STATUS (global)', 'Mann-Whitney', 'global',
              len(df), int(df['efs_event'].sum()),
              f'U={u_stat:.0f}', p_val, f'd={d:.3f}',
              'Step 14')


In [9]:
# P1A.3: KM OS by AGE group (young vs old)
df = merged.dropna(subset=['AGE', 'OS_MONTHS', 'os_event']).copy()
median_age = df['AGE'].median()
df['age_group'] = np.where(df['AGE'] <= median_age, f'Young (≤{median_age:.0f}y)', f'Old (>{median_age:.0f}y)')

young = df[df['age_group'] == f'Young (≤{median_age:.0f}y)']
old = df[df['age_group'] == f'Old (>{median_age:.0f}y)']
p_log = logrank2(young['OS_MONTHS'], young['os_event'], old['OS_MONTHS'], old['os_event'])

km_young = kaplan_meier(young['OS_MONTHS'], young['os_event'])
km_old = kaplan_meier(old['OS_MONTHS'], old['os_event'])

fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2], 
                    subplot_titles=(f'OS by Age Group — Log-rank p={p_log:.4f}', 'Risk table'))
fig = add_km(fig, km_young, f'Young (≤{median_age:.0f}y, n={len(young)})', '#3498db')
fig = add_km(fig, km_old, f'Old (>{median_age:.0f}y, n={len(old)})', '#e74c3c')
# Risk table
for km, name, color in [(km_young, 'Young', '#3498db'), (km_old, 'Old', '#e74c3c')]:
    fig.add_trace(go.Scatter(x=km['t'], y=[km['n'].iloc[0]]*len(km), mode='lines',
                             name=f'{name} at risk', line=dict(color=color, width=0),
                             showlegend=False, hoverinfo='skip'), row=2, col=1)
    fig.add_trace(go.Scatter(x=km['t'], y=km['n'], mode='markers+text',
                             text=km['n'].astype(int), textposition='middle right',
                             name=f'{name} n', line=dict(color=color, width=1),
                             showlegend=False), row=2, col=1)
fig.update_layout(height=500, title='OS by Age Group (global)', template='plotly_white')
fig.update_yaxes(title_text='Survival Probability', row=1, col=1)
fig.update_yaxes(title_text='At Risk', row=2, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
fig.show()

record_result('1A', 'AGE × OS (KM, global)', 'Log-rank', 'global',
              len(df), int(df['os_event'].sum()),
              f'χ²(1)={-2*np.log(p_log):.2f}', p_log, f'median split at {median_age:.0f}y',
              'Step 14')


In [10]:
# P1A.4: KM EFS by AGE group
df = merged.dropna(subset=['AGE', 'EFS_MONTHS', 'efs_event']).copy()
median_age = df['AGE'].median()
df['age_group'] = np.where(df['AGE'] <= median_age, f'Young (≤{median_age:.0f}y)', f'Old (>{median_age:.0f}y)')

young = df[df['age_group'] == f'Young (≤{median_age:.0f}y)']
old = df[df['age_group'] == f'Old (>{median_age:.0f}y)']
p_log = logrank2(young['EFS_MONTHS'], young['efs_event'], old['EFS_MONTHS'], old['efs_event'])

km_young = kaplan_meier(young['EFS_MONTHS'], young['efs_event'])
km_old = kaplan_meier(old['EFS_MONTHS'], old['efs_event'])

fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'EFS by Age Group — Log-rank p={p_log:.4f}', 'Risk table'))
fig = add_km(fig, km_young, f'Young (≤{median_age:.0f}y, n={len(young)})', '#3498db')
fig = add_km(fig, km_old, f'Old (>{median_age:.0f}y, n={len(old)})', '#e74c3c')
fig.update_layout(height=500, title='EFS by Age Group (global)', template='plotly_white')
fig.update_yaxes(title_text='Event-Free Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
fig.show()

record_result('1A', 'AGE × EFS (KM, global)', 'Log-rank', 'global',
              len(df), int(df['efs_event'].sum()),
              f'χ²(1)={-2*np.log(p_log):.2f}', p_log, f'median split at {median_age:.0f}y',
              'Step 14')


In [11]:
# P1A.5: Per-group KM OS by AGE group
from itertools import product

target_os = [g for g in target_groups if 
             merged[(merged['CANCER_GROUP']==g) & merged['AGE'].notna() & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].shape[0] >= 20]

results_1a_os = []
km_data_1a_os = {}
n_groups = len(target_os)
n_cols = min(3, n_groups)
n_rows = int(np.ceil(n_groups / n_cols))

fig = make_subplots(rows=n_rows, cols=n_cols, 
                    subplot_titles=[f'{g[:20]}' for g in target_os],
                    vertical_spacing=0.08, horizontal_spacing=0.06)

colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
idx = 0
for group in target_os:
    gdata = merged[(merged['CANCER_GROUP']==group) & merged['AGE'].notna() 
                   & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].copy()
    if len(gdata) < 20: continue
    med = gdata['AGE'].median()
    gdata['age_g'] = np.where(gdata['AGE'] <= med, f'≤{med:.0f}', f'>{med:.0f}')
    if gdata['age_g'].nunique() < 2: continue
    
    grps = [(gdata[gdata['age_g']==v]['OS_MONTHS'], gdata[gdata['age_g']==v]['os_event']) 
            for v in sorted(gdata['age_g'].unique())]
    p = logrank_multi(grps)
    n_events = int(gdata['os_event'].sum())
    n_total = len(gdata)
    results_1a_os.append({'group': group, 'n': n_total, 'n_events': n_events, 'p_value': p})
    
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    for vi, v in enumerate(sorted(gdata['age_g'].unique())):
        sub = gdata[gdata['age_g']==v]
        km = kaplan_meier(sub['OS_MONTHS'], sub['os_event'])
        fig = add_km(fig, km, f'{group[:20]} — {v} (n={len(sub)})', colors[vi % len(colors)], row=row, col=col)
        fig.update_xaxes(title_text='', row=row, col=col)
        fig.update_yaxes(title_text='', row=row, col=col)
    # Annotate p-value
    fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}',
                       x=0.95, y=0.05, showarrow=False, font=dict(size=10))
    idx += 1

fig.update_layout(height=250*n_rows, title_text='OS by AGE Group per Cancer Group',
                  template='plotly_white', showlegend=True)
fig.show()

# FDR correction on per-group results
if results_1a_os:
    p_vals = [r['p_value'] for r in results_1a_os]
    _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
    for i, r in enumerate(results_1a_os):
        r['q_value'] = q_vals[i]
        record_result('1A', 'AGE × OS (KM, per-group)', 'Log-rank', r['group'],
                      r['n'], r['n_events'], f'χ²(1)={-2*np.log(r["p_value"]):.2f}',
                      r['p_value'], f'q={q_vals[i]:.4f}', 'Step 14')
    print(f"Per-group AGE × OS results (FDR corrected):")
    print(f"{'Group':40s} {'N':6s} {'Events':6s} {'p':8s} {'q':8s}")
    print('-'*80)
    for r in results_1a_os:
        sig = '✅' if r['q_value'] < 0.05 else '❌'
        print(f"{r['group']:40s} {r['n']:6d} {r['n_events']:6d} {r['p_value']:.4f}  {r['q_value']:.4f}  {sig}")


Per-group AGE × OS results (FDR corrected):
Group                                    N      Events p        q       
--------------------------------------------------------------------------------
Adamantinomatous Craniopharyngioma           98      3 0.4643  0.8021  ❌
Atypical Teratoid Rhabdoid Tumor            141     90 0.4740  0.8021  ❌
CNS Embryonal tumor                          24     14 0.1145  0.3058  ❌
Chordoma                                     26     17 0.0016  0.0120  ✅
Choroid plexus tumor                         89     14 0.0325  0.1190  ❌
Diffuse hemispheric glioma                   26     24 0.0011  0.0120  ✅
Diffuse midline glioma                      368    356 0.8021  0.9288  ❌
Dysembryoplastic neuroepithelial tumor       47      0 1.0000  1.0000  ❌
Embryonal tumor with multilayer rosettes     22     16 0.7052  0.9288  ❌
Ependymoma                                  296    126 0.7881  0.9288  ❌
Ewing sarcoma                                34     19 0.3330  0.6912  ❌

In [12]:
# P1A.6: Per-group KM EFS by AGE group
target_efs = [g for g in target_groups if 
              merged[(merged['CANCER_GROUP']==g) & merged['AGE'].notna() & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].shape[0] >= 20]

results_1a_efs = []
n_groups = len(target_efs)
n_cols = min(3, n_groups)
n_rows = max(1, int(np.ceil(n_groups / n_cols)))

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=[f'{g[:20]}' for g in target_efs],
                    vertical_spacing=0.08, horizontal_spacing=0.06)

idx = 0
for group in target_efs:
    gdata = merged[(merged['CANCER_GROUP']==group) & merged['AGE'].notna()
                   & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].copy()
    if len(gdata) < 20: continue
    med = gdata['AGE'].median()
    gdata['age_g'] = np.where(gdata['AGE'] <= med, f'≤{med:.0f}', f'>{med:.0f}')
    if gdata['age_g'].nunique() < 2: continue
    
    grps = [(gdata[gdata['age_g']==v]['EFS_MONTHS'], gdata[gdata['age_g']==v]['efs_event'])
            for v in sorted(gdata['age_g'].unique())]
    p = logrank_multi(grps)
    n_events = int(gdata['efs_event'].sum())
    results_1a_efs.append({'group': group, 'n': len(gdata), 'n_events': n_events, 'p_value': p})
    
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    for vi, v in enumerate(sorted(gdata['age_g'].unique())):
        sub = gdata[gdata['age_g']==v]
        km = kaplan_meier(sub['EFS_MONTHS'], sub['efs_event'])
        fig = add_km(fig, km, f'{group[:20]} — {v} (n={len(sub)})', colors[vi % len(colors)], row=row, col=col)
    fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}',
                       x=0.95, y=0.05, showarrow=False, font=dict(size=10))
    idx += 1

fig.update_layout(height=250*n_rows, title_text='EFS by AGE Group per Cancer Group',
                  template='plotly_white', showlegend=True)
fig.show()

if results_1a_efs:
    p_vals = [r['p_value'] for r in results_1a_efs]
    _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
    for i, r in enumerate(results_1a_efs):
        r['q_value'] = q_vals[i]
        record_result('1A', 'AGE × EFS (KM, per-group)', 'Log-rank', r['group'],
                      r['n'], r['n_events'], f'χ²(1)={-2*np.log(r["p_value"]):.2f}',
                      r['p_value'], f'q={q_vals[i]:.4f}', 'Step 14')
    print(f"Per-group AGE × EFS results (FDR corrected):")
    for r in results_1a_efs:
        sig = '✅' if r['q_value'] < 0.05 else '❌'
        print(f"{r['group']:40s} {r['n']:6d} {r['n_events']:6d} {r['p_value']:.4f}  {r['q_value']:.4f}  {sig}")


Per-group AGE × EFS results (FDR corrected):
Adamantinomatous Craniopharyngioma           98     66 0.5054  0.6540  ❌
Atypical Teratoid Rhabdoid Tumor            141    116 0.8712  0.9127  ❌
CNS Embryonal tumor                          24     20 0.7404  0.9049  ❌
Chordoma                                     26     18 0.0239  0.0699  ❌
Choroid plexus tumor                         88     25 0.8319  0.9127  ❌
Diffuse hemispheric glioma                   26     25 0.0023  0.0125  ✅
Diffuse midline glioma                      276    270 0.3461  0.6345  ❌
Dysembryoplastic neuroepithelial tumor       47     14 0.2944  0.5889  ❌
Embryonal tumor with multilayer rosettes     22     20 0.0001  0.0008  ✅
Ependymoma                                  296    207 0.0208  0.0699  ❌
Ewing sarcoma                                34     20 0.2140  0.5230  ❌
Ganglioglioma                               123     52 0.2427  0.5338  ❌
Glial-neuronal tumor NOS                     47     30 0.4832  0.6540  ❌
High-g

## Section 1B: TUMOR_FRACTION × OS/EFS**Rationale:** Tumor purity (fraction of tumor cells in the sample) may reflect tumor biology — more aggressive tumors may outgrow their stroma (higher purity), or lower purity may indicate a more infiltrative phenotype.**Basic notebook reference:** Step 16 shows TF by CANCER_GROUP and TUMOR_TYPE. This extends to outcome.

In [13]:
# Validation: TUMOR_FRACTION × OS/EFStf_os = merged.dropna(subset=['TUMOR_FRACTION', 'OS_MONTHS', 'os_event'])tf_efs = merged.dropna(subset=['TUMOR_FRACTION', 'EFS_MONTHS', 'efs_event'])print(f"TF × OS: {len(tf_os)} samples ({len(tf_os)/len(merged.dropna(subset=['OS_MONTHS','os_event']))*100:.1f}% of OS samples)")print(f"TF × EFS: {len(tf_efs)} samples")print(f"TF missing overall: {merged['TUMOR_FRACTION'].isna().sum()}/{len(merged)} ({merged['TUMOR_FRACTION'].isna().mean()*100:.1f}%)")

In [14]:
# P1B.1: TF × OS_STATUS (global)
df = merged.dropna(subset=['TUMOR_FRACTION', 'OS_STATUS']).copy()
df['os_label'] = df['OS_STATUS'].str.replace(r'^\d+:', '', regex=True)
living = df[df['os_label']=='LIVING']['TUMOR_FRACTION']
deceased = df[df['os_label']=='DECEASED']['TUMOR_FRACTION']
u_stat, p_val = mannwhitneyu(living, deceased, alternative='two-sided')
d = cliffs_delta(living.values, deceased.values)
fig = go.Figure()
fig.add_trace(go.Box(y=living, name='LIVING', boxmean='sd', marker_color='#2ecc71'))
fig.add_trace(go.Box(y=deceased, name='DECEASED', boxmean='sd', marker_color='#e74c3c'))
fig.update_layout(title=f'TF by OS_STATUS — Mann-Whitney U={u_stat:.0f}, p={p_val:.4f}, d={d:.3f}',
                  yaxis_title='Tumor Fraction', height=450, template='plotly_white')
try:
    fig.show()
except:
    pass
record_result('1B', 'TF × OS_STATUS (global)', 'Mann-Whitney', 'global',
              len(df), int(df['os_label'].eq('DECEASED').sum()),
              f'U={u_stat:.0f}', p_val, f'd={d:.3f}', 'Step 16')

In [15]:
# P1B.2: TF × EFS_STATUS (global)
df = merged.dropna(subset=['TUMOR_FRACTION', 'efs_event']).copy()
no_event = df[df['efs_event']==0]['TUMOR_FRACTION']
event = df[df['efs_event']==1]['TUMOR_FRACTION']
u_stat, p_val = mannwhitneyu(no_event, event, alternative='two-sided')
d = cliffs_delta(no_event.values, event.values)
fig = go.Figure()
fig.add_trace(go.Box(y=no_event, name='No Event', boxmean='sd', marker_color='#3498db'))
fig.add_trace(go.Box(y=event, name='Event', boxmean='sd', marker_color='#e67e22'))
fig.update_layout(title=f'TF by EFS_STATUS — Mann-Whitney U={u_stat:.0f}, p={p_val:.4f}, d={d:.3f}',
                  yaxis_title='Tumor Fraction', height=450, template='plotly_white')
try:
    fig.show()
except:
    pass
record_result('1B', 'TF × EFS_STATUS (global)', 'Mann-Whitney', 'global',
              len(df), int(df['efs_event'].sum()),
              f'U={u_stat:.0f}', p_val, f'd={d:.3f}', 'Step 16')

In [16]:
# P1B.3: KM OS by TF group
df = merged.dropna(subset=['TUMOR_FRACTION', 'OS_MONTHS', 'os_event']).copy()
med = df['TUMOR_FRACTION'].median()
df['tf_group'] = np.where(df['TUMOR_FRACTION'] <= med, f'Low TF (\u2264{med:.2f})', f'High TF (>{med:.2f})')
low = df[df['tf_group']==f'Low TF (\u2264{med:.2f})']
high = df[df['tf_group']==f'High TF (>{med:.2f})']
p_log = logrank2(low['OS_MONTHS'], low['os_event'], high['OS_MONTHS'], high['os_event'])
km_low = kaplan_meier(low['OS_MONTHS'], low['os_event'])
km_high = kaplan_meier(high['OS_MONTHS'], high['os_event'])
fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'OS by TF Group — Log-rank p={p_log:.4f}', 'Risk table'))
fig = add_km(fig, km_low, f'Low TF (n={len(low)})', '#3498db')
fig = add_km(fig, km_high, f'High TF (n={len(high)})', '#e74c3c')
fig.update_layout(height=500, title='OS by Tumor Fraction (global)', template='plotly_white')
fig.update_yaxes(title_text='Survival Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
try:
    fig.show()
except:
    pass
record_result('1B', 'TF × OS (KM, global)', 'Log-rank', 'global',
              len(df), int(df['os_event'].sum()),
              f'\u03c7\u00b2(1)={-2*np.log(p_log):.2f}', p_log, f'median split at {med:.2f}', 'Step 16')

In [17]:
# P1B.4: KM EFS by TF group
df = merged.dropna(subset=['TUMOR_FRACTION', 'EFS_MONTHS', 'efs_event']).copy()
med = df['TUMOR_FRACTION'].median()
df['tf_group'] = np.where(df['TUMOR_FRACTION'] <= med, f'Low TF (\u2264{med:.2f})', f'High TF (>{med:.2f})')
low = df[df['tf_group']==f'Low TF (\u2264{med:.2f})']
high = df[df['tf_group']==f'High TF (>{med:.2f})']
p_log = logrank2(low['EFS_MONTHS'], low['efs_event'], high['EFS_MONTHS'], high['efs_event'])
km_low = kaplan_meier(low['EFS_MONTHS'], low['efs_event'])
km_high = kaplan_meier(high['EFS_MONTHS'], high['efs_event'])
fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'EFS by TF Group — Log-rank p={p_log:.4f}', 'Risk table'))
fig = add_km(fig, km_low, f'Low TF (n={len(low)})', '#3498db')
fig = add_km(fig, km_high, f'High TF (n={len(high)})', '#e74c3c')
fig.update_layout(height=500, title='EFS by Tumor Fraction (global)', template='plotly_white')
fig.update_yaxes(title_text='Event-Free Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
try:
    fig.show()
except:
    pass
record_result('1B', 'TF × EFS (KM, global)', 'Log-rank', 'global',
              len(df), int(df['efs_event'].sum()),
              f'\u03c7\u00b2(1)={-2*np.log(p_log):.2f}', p_log, f'median split at {med:.2f}', 'Step 16')

In [18]:
# P1B.5: Per-group KM OS by TF
target_os_tf = [g for g in target_groups if
    merged[(merged['CANCER_GROUP']==g) & merged['TUMOR_FRACTION'].notna()
           & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].shape[0] >= 20]
results_tf_os = []
n_groups = len(target_os_tf)
if n_groups > 0:
    n_cols = min(3, n_groups); n_rows = int(np.ceil(n_groups / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[f'{g[:20]}' for g in target_os_tf],
                        vertical_spacing=0.08, horizontal_spacing=0.06)
    colors = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6','#1abc9c']
    idx = 0
    for group in target_os_tf:
        gdata = merged[(merged['CANCER_GROUP']==group) & merged['TUMOR_FRACTION'].notna()
                       & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].copy()
        if len(gdata) < 20: continue
        med = gdata['TUMOR_FRACTION'].median()
        gdata['tf_g'] = np.where(gdata['TUMOR_FRACTION'] <= med,
                                 f'\u2264{med:.2f}', f'>{med:.2f}')
        if gdata['tf_g'].nunique() < 2: continue
        grps = [(gdata[gdata['tf_g']==v]['OS_MONTHS'],
                 gdata[gdata['tf_g']==v]['os_event'])
                for v in sorted(gdata['tf_g'].unique())]
        p = logrank_multi(grps)
        results_tf_os.append({'group':group,'n':len(gdata),'n_events':int(gdata['os_event'].sum()),'p_value':p})
        row = idx//n_cols+1; col = idx%n_cols+1
        for vi, v in enumerate(sorted(gdata['tf_g'].unique())):
            sub = gdata[gdata['tf_g']==v]
            km = kaplan_meier(sub['OS_MONTHS'], sub['os_event'])
            fig = add_km(fig, km, f'{group[:20]} — {v} (n={len(sub)})', colors[vi%len(colors)], row=row, col=col)
        fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}',
                           x=0.95, y=0.05, showarrow=False, font=dict(size=10))
        idx += 1
    fig.update_layout(height=250*n_rows,
                      title_text='OS by TF Group per Cancer Group',
                      template='plotly_white', showlegend=True)
    try:
        fig.show()
    except:
        pass
    if results_tf_os:
        p_vals = [r['p_value'] for r in results_tf_os]
        _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
        for i, r in enumerate(results_tf_os):
            record_result('1B', 'TF × OS (KM, per-group)', 'Log-rank', r['group'],
                          r['n'], r['n_events'],
                          f'\u03c7\u00b2={-2*np.log(r["p_value"]):.2f}', r['p_value'],
                          f'q={q_vals[i]:.4f}', 'Step 16')
else:
    print("No cancer groups with n\u226520 for TF \u00d7 OS analysis.")

In [19]:
# P1B.6: Per-group KM EFS by TF
target_efs_tf = [g for g in target_groups if
    merged[(merged['CANCER_GROUP']==g) & merged['TUMOR_FRACTION'].notna()
           & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].shape[0] >= 20]
results_tf_efs = []
n_groups = len(target_efs_tf)
if n_groups > 0:
    n_cols = min(3, n_groups); n_rows = int(np.ceil(n_groups / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[f'{g[:20]}' for g in target_efs_tf],
                        vertical_spacing=0.08, horizontal_spacing=0.06)
    colors = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6','#1abc9c']
    idx = 0
    for group in target_efs_tf:
        gdata = merged[(merged['CANCER_GROUP']==group) & merged['TUMOR_FRACTION'].notna()
                       & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].copy()
        if len(gdata) < 20: continue
        med = gdata['TUMOR_FRACTION'].median()
        gdata['tf_g'] = np.where(gdata['TUMOR_FRACTION'] <= med,
                                 f'\u2264{med:.2f}', f'>{med:.2f}')
        if gdata['tf_g'].nunique() < 2: continue
        grps = [(gdata[gdata['tf_g']==v]['EFS_MONTHS'],
                 gdata[gdata['tf_g']==v]['efs_event'])
                for v in sorted(gdata['tf_g'].unique())]
        p = logrank_multi(grps)
        results_tf_efs.append({'group':group,'n':len(gdata),'n_events':int(gdata['efs_event'].sum()),'p_value':p})
        row = idx//n_cols+1; col = idx%n_cols+1
        for vi, v in enumerate(sorted(gdata['tf_g'].unique())):
            sub = gdata[gdata['tf_g']==v]
            km = kaplan_meier(sub['EFS_MONTHS'], sub['efs_event'])
            fig = add_km(fig, km, f'{group[:20]} — {v} (n={len(sub)})', colors[vi%len(colors)], row=row, col=col)
        fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}',
                           x=0.95, y=0.05, showarrow=False, font=dict(size=10))
        idx += 1
    fig.update_layout(height=250*n_rows,
                      title_text='EFS by TF Group per Cancer Group',
                      template='plotly_white', showlegend=True)
    try:
        fig.show()
    except:
        pass
    if results_tf_efs:
        p_vals = [r['p_value'] for r in results_tf_efs]
        _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
        for i, r in enumerate(results_tf_efs):
            record_result('1B', 'TF × EFS (KM, per-group)', 'Log-rank', r['group'],
                          r['n'], r['n_events'],
                          f'\u03c7\u00b2={-2*np.log(r["p_value"]):.2f}', r['p_value'],
                          f'q={q_vals[i]:.4f}', 'Step 16')
else:
    print("No cancer groups with n\u226520 for TF \u00d7 EFS analysis.")

## Section 1C: TUMOR_PLOIDY × OS/EFS

**Rationale:** Aneuploidy (ploidy ≠ 2) is a hallmark of cancer. Higher ploidy may indicate more genomic instability and potentially worse prognosis.

**Basic notebook reference:** Step 16 shows TP by CANCER_GROUP. This extends to outcome.

In [20]:
# Validation: TUMOR_PLOIDY × OS/EFS
tp_os = merged.dropna(subset=['TUMOR_PLOIDY', 'OS_MONTHS', 'os_event'])
tp_efs = merged.dropna(subset=['TUMOR_PLOIDY', 'EFS_MONTHS', 'efs_event'])
print(f"TP × OS: {len(tp_os)} samples ({len(tp_os)/len(merged.dropna(subset=['OS_MONTHS','os_event']))*100:.1f}% of OS samples)")
print(f"TP × EFS: {len(tp_efs)} samples")
print(f"TP missing overall: {merged['TUMOR_PLOIDY'].isna().sum()}/{len(merged)} ({merged['TUMOR_PLOIDY'].isna().mean()*100:.1f}%)")

TP × OS: 2393 samples (68.8% of OS samples)
TP × EFS: 2282 samples
TP missing overall: 1374/4312 (31.9%)


In [21]:
# P1C.1: TP × OS_STATUS (global)
df = merged.dropna(subset=['TUMOR_PLOIDY', 'OS_STATUS']).copy()
df['os_label'] = df['OS_STATUS'].str.replace(r'^\d+:', '', regex=True)
living = df[df['os_label']=='LIVING']['TUMOR_PLOIDY']
deceased = df[df['os_label']=='DECEASED']['TUMOR_PLOIDY']
u_stat, p_val = mannwhitneyu(living, deceased, alternative='two-sided')
d = cliffs_delta(living.values, deceased.values)

fig = go.Figure()
fig.add_trace(go.Box(y=living, name='LIVING', boxmean='sd', marker_color='#2ecc71'))
fig.add_trace(go.Box(y=deceased, name='DECEASED', boxmean='sd', marker_color='#e74c3c'))
fig.update_layout(title=f'TP by OS_STATUS — Mann-Whitney U={u_stat:.0f}, p={p_val:.4f}, d={d:.3f}',
                  yaxis_title='Tumor Ploidy', height=450, template='plotly_white')
fig.show()
record_result('1C', 'TP × OS_STATUS (global)', 'Mann-Whitney', 'global',
              len(df), int(df['os_label'].eq('DECEASED').sum()),
              f'U={u_stat:.0f}', p_val, f'd={d:.3f}', 'Step 16')

In [22]:
# P1C.2: TP × EFS_STATUS (global)
df = merged.dropna(subset=['TUMOR_PLOIDY', 'efs_event']).copy()
no_event = df[df['efs_event']==0]['TUMOR_PLOIDY']
event = df[df['efs_event']==1]['TUMOR_PLOIDY']
u_stat, p_val = mannwhitneyu(no_event, event, alternative='two-sided')
d = cliffs_delta(no_event.values, event.values)

fig = go.Figure()
fig.add_trace(go.Box(y=no_event, name='No Event', boxmean='sd', marker_color='#3498db'))
fig.add_trace(go.Box(y=event, name='Event', boxmean='sd', marker_color='#e67e22'))
fig.update_layout(title=f'TP by EFS_STATUS — Mann-Whitney U={u_stat:.0f}, p={p_val:.4f}, d={d:.3f}',
                  yaxis_title='Tumor Ploidy', height=450, template='plotly_white')
fig.show()
record_result('1C', 'TP × EFS_STATUS (global)', 'Mann-Whitney', 'global',
              len(df), int(df['efs_event'].sum()),
              f'U={u_stat:.0f}', p_val, f'd={d:.3f}', 'Step 16')

In [23]:
# P1C.3: KM OS by TP (diploid ≈2 vs aneuploid)
df = merged.dropna(subset=['TUMOR_PLOIDY', 'OS_MONTHS', 'os_event']).copy()
df['tp_group'] = np.where((df['TUMOR_PLOIDY'] >= 1.8) & (df['TUMOR_PLOIDY'] <= 2.2), 'Diploid (~2)', 'Aneuploid')
if df['tp_group'].nunique() >= 2:
    diploid = df[df['tp_group']=='Diploid (~2)']
    aneuploid = df[df['tp_group']=='Aneuploid']
    p_log = logrank2(diploid['OS_MONTHS'], diploid['os_event'], aneuploid['OS_MONTHS'], aneuploid['os_event'])
    km_dip = kaplan_meier(diploid['OS_MONTHS'], diploid['os_event'])
    km_aneu = kaplan_meier(aneuploid['OS_MONTHS'], aneuploid['os_event'])
    fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                        subplot_titles=(f'OS by TP — Log-rank p={p_log:.4f}', 'Risk table'))
    fig = add_km(fig, km_dip, f'Diploid (n={len(diploid)})', '#2ecc71')
    fig = add_km(fig, km_aneu, f'Aneuploid (n={len(aneuploid)})', '#e74c3c')
    fig.update_layout(height=500, title='OS by Tumor Ploidy (global)', template='plotly_white')
    fig.update_yaxes(title_text='Survival Probability', row=1, col=1)
    fig.update_xaxes(title_text='Months', row=2, col=1)
    fig.show()
    record_result('1C', 'TP × OS (KM, global)', 'Log-rank', 'global',
                  len(df), int(df['os_event'].sum()),
                  f'χ²(1)={-2*np.log(p_log):.2f}', p_log, 'diploid vs aneuploid', 'Step 16')
else:
    print("Not enough diploid/aneuploid groups for KM comparison.")

In [24]:
# P1C.4: KM EFS by TP (diploid vs aneuploid)
df = merged.dropna(subset=['TUMOR_PLOIDY', 'EFS_MONTHS', 'efs_event']).copy()
df['tp_group'] = np.where((df['TUMOR_PLOIDY'] >= 1.8) & (df['TUMOR_PLOIDY'] <= 2.2), 'Diploid (~2)', 'Aneuploid')
if df['tp_group'].nunique() >= 2:
    diploid = df[df['tp_group']=='Diploid (~2)']
    aneuploid = df[df['tp_group']=='Aneuploid']
    p_log = logrank2(diploid['EFS_MONTHS'], diploid['efs_event'], aneuploid['EFS_MONTHS'], aneuploid['efs_event'])
    km_dip = kaplan_meier(diploid['EFS_MONTHS'], diploid['efs_event'])
    km_aneu = kaplan_meier(aneuploid['EFS_MONTHS'], aneuploid['efs_event'])
    fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                        subplot_titles=(f'EFS by TP — Log-rank p={p_log:.4f}', 'Risk table'))
    fig = add_km(fig, km_dip, f'Diploid (n={len(diploid)})', '#2ecc71')
    fig = add_km(fig, km_aneu, f'Aneuploid (n={len(aneuploid)})', '#e74c3c')
    fig.update_layout(height=500, title='EFS by Tumor Ploidy (global)', template='plotly_white')
    fig.update_yaxes(title_text='Event-Free Probability', row=1, col=1)
    fig.update_xaxes(title_text='Months', row=2, col=1)
    fig.show()
    record_result('1C', 'TP × EFS (KM, global)', 'Log-rank', 'global',
                  len(df), int(df['efs_event'].sum()),
                  f'χ²(1)={-2*np.log(p_log):.2f}', p_log, 'diploid vs aneuploid', 'Step 16')
else:
    print("Not enough diploid/aneuploid groups for KM comparison.")

In [25]:
# P1C.5: Per-group KM OS by TP (diploid vs aneuploid)
target_os_tp = [g for g in target_groups if
    merged[(merged['CANCER_GROUP']==g) & merged['TUMOR_PLOIDY'].notna() & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].shape[0] >= 20]

results_tp_os = []
n_groups = len(target_os_tp)
if n_groups > 0:
    n_cols = min(3, n_groups); n_rows = int(np.ceil(n_groups / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=[f'{g[:20]}' for g in target_os_tp],
                        vertical_spacing=0.08, horizontal_spacing=0.06)
    colors = ['#2ecc71','#e74c3c','#3498db','#f39c12']
    idx = 0
    for group in target_os_tp:
        gdata = merged[(merged['CANCER_GROUP']==group) & merged['TUMOR_PLOIDY'].notna()
                       & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].copy()
        if len(gdata) < 20: continue
        gdata['tp_g'] = np.where((gdata['TUMOR_PLOIDY']>=1.8)&(gdata['TUMOR_PLOIDY']<=2.2), 'Diploid', 'Aneuploid')
        if gdata['tp_g'].nunique() < 2: continue
        grps = [(gdata[gdata['tp_g']==v]['OS_MONTHS'], gdata[gdata['tp_g']==v]['os_event']) for v in sorted(gdata['tp_g'].unique())]
        p = logrank_multi(grps)
        results_tp_os.append({'group':group,'n':len(gdata),'n_events':int(gdata['os_event'].sum()),'p_value':p})
        row = idx//n_cols+1; col = idx%n_cols+1
        for vi, v in enumerate(sorted(gdata['tp_g'].unique())):
            sub = gdata[gdata['tp_g']==v]
            km = kaplan_meier(sub['OS_MONTHS'], sub['os_event'])
            fig = add_km(fig, km, f'{group[:20]} — {v} (n={len(sub)})', colors[vi%len(colors)], row=row, col=col)
        fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}', x=0.95, y=0.05, showarrow=False, font=dict(size=10))
        idx += 1
    fig.update_layout(height=250*n_rows, title_text='OS by TP (Diploid vs Aneuploid) per Cancer Group', template='plotly_white', showlegend=True)
    fig.show()
    if results_tp_os:
        p_vals = [r['p_value'] for r in results_tp_os]
        _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
        for i, r in enumerate(results_tp_os):
            record_result('1C', 'TP × OS (KM, per-group)', 'Log-rank', r['group'],
                          r['n'], r['n_events'], f'χ²={-2*np.log(r["p_value"]):.2f}', r['p_value'],
                          f'q={q_vals[i]:.4f}', 'Step 16')
else:
    print("No cancer groups with n≥20 for TP × OS analysis.")

In [26]:
# P1C.6: Per-group KM EFS by TP (diploid vs aneuploid)
target_efs_tp = [g for g in target_groups if
    merged[(merged['CANCER_GROUP']==g) & merged['TUMOR_PLOIDY'].notna() & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].shape[0] >= 20]

results_tp_efs = []
n_groups = len(target_efs_tp)
if n_groups > 0:
    n_cols = min(3, n_groups); n_rows = int(np.ceil(n_groups / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=[f'{g[:20]}' for g in target_efs_tp],
                        vertical_spacing=0.08, horizontal_spacing=0.06)
    colors = ['#2ecc71','#e74c3c','#3498db','#f39c12']
    idx = 0
    for group in target_efs_tp:
        gdata = merged[(merged['CANCER_GROUP']==group) & merged['TUMOR_PLOIDY'].notna()
                       & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].copy()
        if len(gdata) < 20: continue
        gdata['tp_g'] = np.where((gdata['TUMOR_PLOIDY']>=1.8)&(gdata['TUMOR_PLOIDY']<=2.2), 'Diploid', 'Aneuploid')
        if gdata['tp_g'].nunique() < 2: continue
        grps = [(gdata[gdata['tp_g']==v]['EFS_MONTHS'], gdata[gdata['tp_g']==v]['efs_event']) for v in sorted(gdata['tp_g'].unique())]
        p = logrank_multi(grps)
        results_tp_efs.append({'group':group,'n':len(gdata),'n_events':int(gdata['efs_event'].sum()),'p_value':p})
        row = idx//n_cols+1; col = idx%n_cols+1
        for vi, v in enumerate(sorted(gdata['tp_g'].unique())):
            sub = gdata[gdata['tp_g']==v]
            km = kaplan_meier(sub['EFS_MONTHS'], sub['efs_event'])
            fig = add_km(fig, km, f'{group[:20]} — {v} (n={len(sub)})', colors[vi%len(colors)], row=row, col=col)
        fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}', x=0.95, y=0.05, showarrow=False, font=dict(size=10))
        idx += 1
    fig.update_layout(height=250*n_rows, title_text='EFS by TP (Diploid vs Aneuploid) per Cancer Group', template='plotly_white', showlegend=True)
    fig.show()
    if results_tp_efs:
        p_vals = [r['p_value'] for r in results_tp_efs]
        _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
        for i, r in enumerate(results_tp_efs):
            record_result('1C', 'TP × EFS (KM, per-group)', 'Log-rank', r['group'],
                          r['n'], r['n_events'], f'χ²={-2*np.log(r["p_value"]):.2f}', r['p_value'],
                          f'q={q_vals[i]:.4f}', 'Step 16')
else:
    print("No cancer groups with n≥20 for TP × EFS analysis.")

## Section 1D: SEX × OS/EFS

**Rationale:** Sex differences in cancer incidence and outcome are well-documented in adults. Less is known in pediatric brain tumors.

**Basic notebook reference:** Step 15 tests SEX × CANCER_GROUP (sex bias in group composition). This extends to outcome.


In [27]:
# Validation: SEX × OS/EFS
sex_os = merged.dropna(subset=['SEX', 'OS_STATUS'])
sex_efs = merged.dropna(subset=['SEX', 'efs_event'])
ct_os = pd.crosstab(sex_os['SEX'], sex_os['OS_STATUS'].str.replace(r'^\d+:', '', regex=True))
print(f"SEX × OS: {len(sex_os)} samples")
print(f"Contingency table:\n{ct_os}")
print(f"SEX × EFS: {len(sex_efs)} samples")
print(f"SEX missing: {merged['SEX'].isna().sum()}/{len(merged)}")


SEX × OS: 3899 samples
Contingency table:
OS_STATUS     DECEASED  LIVING
SEX                           
Female             647    1136
Male               697    1416
Not Reported         3       0
SEX × EFS: 3826 samples
SEX missing: 19/4312


In [28]:
# P1D.1: SEX × OS_STATUS (global) -- stacked bar + chi-squared
df = merged.dropna(subset=['SEX', 'OS_STATUS']).copy()
df['os_label'] = df['OS_STATUS'].str.replace(r'^\d+:', '', regex=True)
ct = pd.crosstab(df['SEX'], df['os_label'])
chi2_stat, p_val, dof, expected = chi2_contingency(ct)
v = cramers_v(ct)

fig = px.histogram(df, x='SEX', color='os_label', barmode='group',
                   title=f'SEX × OS_STATUS — χ²={chi2_stat:.2f}, p={p_val:.4f}, V={v:.3f}',
                   color_discrete_map={'LIVING':'#2ecc71','DECEASED':'#e74c3c'},
                   labels={'os_label':'OS Status', 'count':'Number of Patients'})
fig.update_layout(template='plotly_white', height=450)
fig.show()
record_result('1D', 'SEX × OS_STATUS (global)', 'Chi-squared', 'global',
              len(df), int(df['os_label'].eq('DECEASED').sum()),
              f'χ²={chi2_stat:.2f}', p_val, f'V={v:.3f}', 'Step 15')


In [29]:
# P1D.2: SEX × EFS_STATUS (global)
df = merged.dropna(subset=['SEX', 'efs_event']).copy()
df['efs_label'] = df['efs_event'].map({0:'No Event', 1:'Event'})
ct = pd.crosstab(df['SEX'], df['efs_label'])
chi2_stat, p_val, dof, expected = chi2_contingency(ct)
v = cramers_v(ct)

fig = px.histogram(df, x='SEX', color='efs_label', barmode='group',
                   title=f'SEX × EFS_STATUS — χ²={chi2_stat:.2f}, p={p_val:.4f}, V={v:.3f}',
                   color_discrete_map={'No Event':'#3498db','Event':'#e67e22'},
                   labels={'efs_label':'EFS Status', 'count':'Number of Patients'})
fig.update_layout(template='plotly_white', height=450)
fig.show()
record_result('1D', 'SEX × EFS_STATUS (global)', 'Chi-squared', 'global',
              len(df), int(df['efs_event'].sum()),
              f'χ²={chi2_stat:.2f}', p_val, f'V={v:.3f}', 'Step 15')


In [30]:
# P1D.3: KM OS by SEX
df = merged.dropna(subset=['SEX', 'OS_MONTHS', 'os_event']).copy()
male = df[df['SEX']=='Male']
female = df[df['SEX']=='Female']
p_log = logrank2(male['OS_MONTHS'], male['os_event'], female['OS_MONTHS'], female['os_event'])

km_male = kaplan_meier(male['OS_MONTHS'], male['os_event'])
km_female = kaplan_meier(female['OS_MONTHS'], female['os_event'])

fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'OS by SEX — Log-rank p={p_log:.4f}', 'Risk table'))
fig = add_km(fig, km_male, f'Male (n={len(male)})', '#3498db')
fig = add_km(fig, km_female, f'Female (n={len(female)})', '#e74c3c')
fig.update_layout(height=500, title='OS by Sex (global)', template='plotly_white')
fig.update_yaxes(title_text='Survival Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
fig.show()
record_result('1D', 'SEX × OS (KM, global)', 'Log-rank', 'global',
              len(df), int(df['os_event'].sum()),
              f'χ²(1)={-2*np.log(p_log):.2f}', p_log, 'Male vs Female', 'Step 15')


In [31]:
# P1D.4: KM EFS by SEX
df = merged.dropna(subset=['SEX', 'EFS_MONTHS', 'efs_event']).copy()
male = df[df['SEX']=='Male']
female = df[df['SEX']=='Female']
p_log = logrank2(male['EFS_MONTHS'], male['efs_event'], female['EFS_MONTHS'], female['efs_event'])

km_male = kaplan_meier(male['EFS_MONTHS'], male['efs_event'])
km_female = kaplan_meier(female['EFS_MONTHS'], female['efs_event'])

fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'EFS by SEX — Log-rank p={p_log:.4f}', 'Risk table'))
fig = add_km(fig, km_male, f'Male (n={len(male)})', '#3498db')
fig = add_km(fig, km_female, f'Female (n={len(female)})', '#e74c3c')
fig.update_layout(height=500, title='EFS by Sex (global)', template='plotly_white')
fig.update_yaxes(title_text='Event-Free Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
fig.show()
record_result('1D', 'SEX × EFS (KM, global)', 'Log-rank', 'global',
              len(df), int(df['efs_event'].sum()),
              f'χ²(1)={-2*np.log(p_log):.2f}', p_log, 'Male vs Female', 'Step 15')


In [32]:
# P1D.5: Per-group KM OS by SEX
target_sex_os = [g for g in target_groups if
    merged[(merged['CANCER_GROUP']==g) & merged['SEX'].notna() & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].shape[0] >= 20]

results_sex_os = []
n_groups = len(target_sex_os)
if n_groups > 0:
    n_cols = min(3, n_groups); n_rows = int(np.ceil(n_groups / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=[f'{g[:20]}' for g in target_sex_os],
                        vertical_spacing=0.08, horizontal_spacing=0.06)
    idx = 0
    for group in target_sex_os:
        gdata = merged[(merged['CANCER_GROUP']==group) & merged['SEX'].notna()
                       & merged['OS_MONTHS'].notna() & merged['os_event'].notna()].copy()
        if len(gdata) < 20: continue
        sexes = [v for v in gdata['SEX'].unique() if v in ['Male','Female']]
        if len(sexes) < 2: continue
        grps = [(gdata[gdata['SEX']==v]['OS_MONTHS'], gdata[gdata['SEX']==v]['os_event']) for v in sexes]
        p = logrank_multi(grps)
        results_sex_os.append({'group':group,'n':len(gdata),'n_events':int(gdata['os_event'].sum()),'p_value':p})
        row = idx//n_cols+1; col = idx%n_cols+1
        colors_sex = {'Male':'#3498db','Female':'#e74c3c'}
        for v in sexes:
            sub = gdata[gdata['SEX']==v]
            km = kaplan_meier(sub['OS_MONTHS'], sub['os_event'])
            fig = add_km(fig, km, f'{group[:20]} — {v} (n={len(sub)})', colors_sex.get(v, '#999'), row=row, col=col)
        fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}', x=0.95, y=0.05, showarrow=False, font=dict(size=10))
        idx += 1
    fig.update_layout(height=250*n_rows, title_text='OS by SEX per Cancer Group', template='plotly_white', showlegend=True)
    fig.show()
    if results_sex_os:
        p_vals = [r['p_value'] for r in results_sex_os]
        _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
        for i, r in enumerate(results_sex_os):
            record_result('1D', 'SEX × OS (KM, per-group)', 'Log-rank', r['group'],
                          r['n'], r['n_events'], f'χ²={-2*np.log(r["p_value"]):.2f}', r['p_value'],
                          f'q={q_vals[i]:.4f}', 'Step 15')
else:
    print("No cancer groups with n≥20 for SEX × OS analysis.")


In [33]:
# P1D.6: Per-group KM EFS by SEX
target_sex_efs = [g for g in target_groups if
    merged[(merged['CANCER_GROUP']==g) & merged['SEX'].notna() & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].shape[0] >= 20]

results_sex_efs = []
n_groups = len(target_sex_efs)
if n_groups > 0:
    n_cols = min(3, n_groups); n_rows = int(np.ceil(n_groups / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=[f'{g[:20]}' for g in target_sex_efs],
                        vertical_spacing=0.08, horizontal_spacing=0.06)
    idx = 0
    for group in target_sex_efs:
        gdata = merged[(merged['CANCER_GROUP']==group) & merged['SEX'].notna()
                       & merged['EFS_MONTHS'].notna() & merged['efs_event'].notna()].copy()
        if len(gdata) < 20: continue
        sexes = [v for v in gdata['SEX'].unique() if v in ['Male','Female']]
        if len(sexes) < 2: continue
        grps = [(gdata[gdata['SEX']==v]['EFS_MONTHS'], gdata[gdata['SEX']==v]['efs_event']) for v in sexes]
        p = logrank_multi(grps)
        results_sex_efs.append({'group':group,'n':len(gdata),'n_events':int(gdata['efs_event'].sum()),'p_value':p})
        row = idx//n_cols+1; col = idx%n_cols+1
        colors_sex = {'Male':'#3498db','Female':'#e74c3c'}
        for v in sexes:
            sub = gdata[gdata['SEX']==v]
            km = kaplan_meier(sub['EFS_MONTHS'], sub['efs_event'])
            fig = add_km(fig, km, f'{group[:20]} — {v} (n={len(sub)})', colors_sex.get(v, '#999'), row=row, col=col)
        fig.add_annotation(text=f'p={p:.4f}', xref=f'x{idx+1}', yref=f'y{idx+1}', x=0.95, y=0.05, showarrow=False, font=dict(size=10))
        idx += 1
    fig.update_layout(height=250*n_rows, title_text='EFS by SEX per Cancer Group', template='plotly_white', showlegend=True)
    fig.show()
    if results_sex_efs:
        p_vals = [r['p_value'] for r in results_sex_efs]
        _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
        for i, r in enumerate(results_sex_efs):
            record_result('1D', 'SEX × EFS (KM, per-group)', 'Log-rank', r['group'],
                          r['n'], r['n_events'], f'χ²={-2*np.log(r["p_value"]):.2f}', r['p_value'],
                          f'q={q_vals[i]:.4f}', 'Step 15')
else:
    print("No cancer groups with n≥20 for SEX × EFS analysis.")


## Section 1E: CANCER_PREDISPOSITIONS × OS/EFS

**Rationale:** Patients with known cancer predisposition syndromes may have different tumor biology and treatment responses.

**Basic notebook reference:** Step 17 shows predisposition × OS (global binary). This extends with per-type breakdown and EFS.
**Note on log-rank tests in this section:** P1E.1–P1E.2 use a *binary* log-rank test (With vs Without predisposition, comparing two groups directly). P1E.3–P1E.4 use a *multi-group* log-rank test that compares all predisposition types simultaneously. The multi-group test answers whether there is any difference across types, while the binary test asks a simpler With/Without question.


In [34]:
# Validation: Predisposition × OS/EFS
pred_os = merged.dropna(subset=['OS_MONTHS', 'os_event']).copy()
pred_os['has_pred'] = ~pred_os['CANCER_PREDISPOSITIONS'].isin(['No predisposition', 'Unknown'])
print(f"Patients with predisposition data: {len(pred_os)}")
print(f"  With predisposition: {pred_os['has_pred'].sum()} ({pred_os['has_pred'].mean()*100:.1f}%)")
print(f"  Without: {(~pred_os['has_pred']).sum()}")
print(f"\nTop predisposition types:")
print(pred_os[~pred_os['CANCER_PREDISPOSITIONS'].isin(['No predisposition', 'Unknown'])]['CANCER_PREDISPOSITIONS'].value_counts().head(10).to_string())


Patients with predisposition data: 3476
  With predisposition: 349 (10.0%)
  Without: 3127

Top predisposition types:
CANCER_PREDISPOSITIONS
Neurofibromatosis, Type 1 (NF-1)                                                                                                 113
Li-Fraumeni syndrome (TP53)                                                                                                       62
Other inherited conditions NOS                                                                                                    59
Neurofibromatosis, Type 2 (NF-2)                                                                                                  40
Tuberous Sclerosis (TSC1, TSC2)                                                                                                   12
Lynch Syndrome (PMS2, MLH1, MSH2, MSH6)                                                                                           10
Von Hippel-Lindau (VHL)                                      

In [35]:
# P1E.1: KM OS by predisposition (binary)
df = merged.dropna(subset=['OS_MONTHS', 'os_event']).copy()
df['has_pred'] = ~df['CANCER_PREDISPOSITIONS'].isin(['No predisposition', 'Unknown'])
with_pred = df[df['has_pred']]
without_pred = df[~df['has_pred']]
p_log = logrank2(with_pred['OS_MONTHS'], with_pred['os_event'], without_pred['OS_MONTHS'], without_pred['os_event'])

km_with = kaplan_meier(with_pred['OS_MONTHS'], with_pred['os_event'])
km_without = kaplan_meier(without_pred['OS_MONTHS'], without_pred['os_event'])

fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'OS by Predisposition Status — Log-rank p={p_log:.4f}', 'Risk table'))
fig = add_km(fig, km_with, f'With Predisposition (n={len(with_pred)})', '#e74c3c')
fig = add_km(fig, km_without, f'No Predisposition (n={len(without_pred)})', '#3498db')
fig.update_layout(height=500, title='OS by Cancer Predisposition (binary)', template='plotly_white')
fig.update_yaxes(title_text='Survival Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
fig.show()
record_result('1E', 'Predisposition × OS (binary, KM)', 'Log-rank', 'global',
              len(df), int(df['os_event'].sum()),
              f'\u03c7²(1)={-2*np.log(p_log):.2f}', p_log, 'With vs Without', 'Step 17')


In [36]:
# P1E.2: KM EFS by predisposition (binary)
df = merged.dropna(subset=['EFS_MONTHS', 'efs_event']).copy()
df['has_pred'] = ~df['CANCER_PREDISPOSITIONS'].isin(['No predisposition', 'Unknown'])
with_pred = df[df['has_pred']]
without_pred = df[~df['has_pred']]
p_log = logrank2(with_pred['EFS_MONTHS'], with_pred['efs_event'], without_pred['EFS_MONTHS'], without_pred['efs_event'])

km_with = kaplan_meier(with_pred['EFS_MONTHS'], with_pred['efs_event'])
km_without = kaplan_meier(without_pred['EFS_MONTHS'], without_pred['efs_event'])

fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                    subplot_titles=(f'EFS by Predisposition Status — Log-rank p={p_log:.4f}', 'Risk table'))
fig = add_km(fig, km_with, f'With Predisposition (n={len(with_pred)})', '#e74c3c')
fig = add_km(fig, km_without, f'No Predisposition (n={len(without_pred)})', '#3498db')
fig.update_layout(height=500, title='EFS by Cancer Predisposition (binary)', template='plotly_white')
fig.update_yaxes(title_text='Event-Free Probability', row=1, col=1)
fig.update_xaxes(title_text='Months', row=2, col=1)
fig.show()
record_result('1E', 'Predisposition × EFS (binary, KM)', 'Log-rank', 'global',
              len(df), int(df['efs_event'].sum()),
              f'\u03c7²(1)={-2*np.log(p_log):.2f}', p_log, 'With vs Without', 'Step 17')


In [37]:
# P1E.3: KM OS by predisposition type (top types)
df = merged.dropna(subset=['OS_MONTHS', 'os_event']).copy()
# Group top predispositions
top_preds = df[~df['CANCER_PREDISPOSITIONS'].isin(['No predisposition', 'Unknown'])]['CANCER_PREDISPOSITIONS'].value_counts().head(5).index.tolist()
df['pred_group'] = df['CANCER_PREDISPOSITIONS'].apply(
    lambda x: x if x in top_preds else ('Other' if x not in ['No predisposition', 'Unknown'] else 'No predisposition'))
pred_groups = df['pred_group'].unique()
pred_groups = [g for g in ['No predisposition'] + [g for g in pred_groups if g != 'No predisposition']]

if len(pred_groups) >= 2:
    grps = [(df[df['pred_group']==v]['OS_MONTHS'], df[df['pred_group']==v]['os_event']) for v in pred_groups]
    p_global = logrank_multi(grps)
    
    fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                        subplot_titles=(f'OS by Predisposition Type — Global log-rank p={p_global:.4f}', 'Risk table'))
    colors = px.colors.qualitative.Set1[:len(pred_groups)]
    for i, v in enumerate(pred_groups):
        sub = df[df['pred_group']==v]
        if len(sub) < 5: continue
        km = kaplan_meier(sub['OS_MONTHS'], sub['os_event'])
        fig = add_km(fig, km, f'{v[:25]} (n={len(sub)})', colors[i % len(colors)])
    fig.update_layout(height=500, title='OS by Predisposition Type', template='plotly_white')
    fig.update_yaxes(title_text='Survival Probability', row=1, col=1)
    fig.update_xaxes(title_text='Months', row=2, col=1)
    fig.show()
    record_result('1E', 'Predisposition type × OS (KM)', 'Log-rank (multi)', 'global',
                  len(df), int(df['os_event'].sum()),
                  f'\u03c7²={-2*np.log(p_global):.2f}', p_global, f'{len(pred_groups)} groups', 'Step 17')


In [38]:
# P1E.4: KM EFS by predisposition type
df = merged.dropna(subset=['EFS_MONTHS', 'efs_event']).copy()
top_preds = df[~df['CANCER_PREDISPOSITIONS'].isin(['No predisposition', 'Unknown'])]['CANCER_PREDISPOSITIONS'].value_counts().head(5).index.tolist()
df['pred_group'] = df['CANCER_PREDISPOSITIONS'].apply(
    lambda x: x if x in top_preds else ('Other' if x not in ['No predisposition', 'Unknown'] else 'No predisposition'))
pred_groups = df['pred_group'].unique()
pred_groups = [g for g in ['No predisposition'] + [g for g in pred_groups if g != 'No predisposition']]

if len(pred_groups) >= 2:
    grps = [(df[df['pred_group']==v]['EFS_MONTHS'], df[df['pred_group']==v]['efs_event']) for v in pred_groups]
    p_global = logrank_multi(grps)
    
    fig = make_subplots(rows=2, cols=1, row_heights=[0.8, 0.2],
                        subplot_titles=(f'EFS by Predisposition Type — Global log-rank p={p_global:.4f}', 'Risk table'))
    colors = px.colors.qualitative.Set1[:len(pred_groups)]
    for i, v in enumerate(pred_groups):
        sub = df[df['pred_group']==v]
        if len(sub) < 5: continue
        km = kaplan_meier(sub['EFS_MONTHS'], sub['efs_event'])
        fig = add_km(fig, km, f'{v[:25]} (n={len(sub)})', colors[i % len(colors)])
    fig.update_layout(height=500, title='EFS by Predisposition Type', template='plotly_white')
    fig.update_yaxes(title_text='Event-Free Probability', row=1, col=1)
    fig.update_xaxes(title_text='Months', row=2, col=1)
    fig.show()
    record_result('1E', 'Predisposition type × EFS (KM)', 'Log-rank (multi)', 'global',
                  len(df), int(df['efs_event'].sum()),
                  f'\u03c7²={-2*np.log(p_global):.2f}', p_global, f'{len(pred_groups)} groups', 'Step 17')


In [39]:
# P1E.5: Predisposition type frequency with outcome composition
df = merged.dropna(subset=['OS_STATUS']).copy()
df['os_label'] = df['OS_STATUS'].str.replace(r'^\d+:', '', regex=True)
pred_counts = df['CANCER_PREDISPOSITIONS'].value_counts()
top_n = pred_counts.head(10)
fig = px.bar(x=top_n.values, y=top_n.index, orientation='h',
             title='Top 10 Cancer Predispositions (all patients)',
             labels={'x':'Number of Patients', 'y':''},
             text=top_n.values, color=top_n.values,
             color_continuous_scale='Blues')
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white', height=400)
fig.show()


## Results Summary

All statistical tests performed in this notebook are consolidated below, with significance flags and effect sizes.


In [40]:
# Build summary table
if results_rows:
    summary = pd.DataFrame(results_rows)
    summary = summary.sort_values(['Phase', 'p_value'])
    
    # Add FDR column (within each Phase family)
    summary['FDR_BH'] = np.nan
    for phase in summary['Phase'].unique():
        mask = summary['Phase'] == phase
        p_vals = summary.loc[mask, 'p_value'].values
        if len(p_vals) > 0:
            _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
            summary.loc[mask, 'FDR_BH'] = q_vals
    
    # Format for display
    display_cols = ['Phase', 'Comparison', 'Test', 'Group', 'N', 'N_events', 
                    'Statistic', 'p_value', 'FDR_BH', 'Significant', 'Effect_Size', 'Basic_Ref']
    display_df = summary[display_cols].copy()
    display_df['FDR_BH'] = display_df['FDR_BH'].round(4)
    
    print(f"\n{'='*100}")
    print(f"SURVIVAL ANALYSIS — RESULTS SUMMARY")
    print(f"{'='*100}")
    print(f"Total statistical tests: {len(display_df)}")
    sig_raw = (display_df['p_value'] < 0.05).sum()
    sig_fdr = (display_df['FDR_BH'] < 0.05).sum()
    print(f"Significant (raw p < 0.05): {sig_raw} ({sig_raw/len(display_df)*100:.1f}%)")
    print(f"Significant (FDR < 0.05): {sig_fdr} ({sig_fdr/len(display_df)*100:.1f}%)")
    print(f"{'='*100}\n")
    
    from IPython.display import display as ipy_display
    ipy_display(display_df)
    
    # Save
    import os
    out_dir = '/home/alon/menow_home_ass/notebooks/survival_analysis'
    os.makedirs(out_dir, exist_ok=True)
    summary.to_csv(f'{out_dir}/survival_analysis_results.csv', index=False)
    print(f"\nSaved: {out_dir}/survival_analysis_results.csv")
    
    # Significance by section
    print(f"\nSignificance count by section:")
    for phase in sorted(summary['Phase'].unique()):
        sub = summary[summary['Phase']==phase]
        n_sig = (sub['p_value'] < 0.05).sum()
        n_sig_fdr = (sub['FDR_BH'] < 0.05).sum() if sub['FDR_BH'].notna().any() else 0
        print(f"  Phase {phase}: {len(sub)} tests, {n_sig} significant (raw), {n_sig_fdr} significant (FDR)")
else:
    print("No results recorded. Something went wrong.")



SURVIVAL ANALYSIS — RESULTS SUMMARY
Total statistical tests: 168
Significant (raw p < 0.05): 39 (23.2%)
Significant (FDR < 0.05): 22 (13.1%)



,Phase,Comparison,Test,Group,N,N_events,Statistic,p_value,FDR_BH,Significant,Effect_Size,Basic_Ref
1,1A,AGE × EFS_STATUS (global),Mann-Whitney,global,3839,2235,U=2081908,0.0000,0.0000,🟢🟢🟢 p<0.001,d=0.161,Step 14
3,1A,"AGE × EFS (KM, global)",Log-rank,global,3358,2165,χ²(1)=52.17,0.0000,0.0000,🟢🟢🟢 p<0.001,median split at 7y,Step 14
40,1A,"AGE × EFS (KM, per-group)",Log-rank,Low-grade glioma,710,369,χ²(1)=27.31,0.0000,0.0000,🟢🟢🟢 p<0.001,q=0.0000,Step 14
34,1A,"AGE × EFS (KM, per-group)",Log-rank,Embryonal tumor with multilayer rosettes,22,20,χ²(1)=19.14,0.0001,0.0010,🟢🟢🟢 p<0.001,q=0.0008,Step 14
39,1A,"AGE × EFS (KM, per-group)",Log-rank,High-grade glioma,388,348,χ²(1)=17.86,0.0001,0.0010,🟢🟢🟢 p<0.001,q=0.0010,Step 14
...,...,...,...,...,...,...,...,...,...,...,...,...
131,1D,"SEX × OS (KM, per-group)",Log-rank,Ganglioglioma,123,0,χ²=-0.00,1.0000,1.0000,❌ NS,q=1.0000,Step 15
166,1E,Predisposition type × OS (KM),Log-rank (multi),global,3476,1353,χ²=45.13,0.0000,0.0000,🟢🟢🟢 p<0.001,7 groups,Step 17
167,1E,Predisposition type × EFS (KM),Log-rank (multi),global,3359,2166,χ²=45.54,0.0000,0.0000,🟢🟢🟢 p<0.001,7 groups,Step 17
164,1E,"Predisposition × OS (binary, KM)",Log-rank,global,3476,1353,χ²(1)=12.55,0.0019,0.0025,🟢🟢 p<0.01,With vs Without,Step 17



Saved: /home/alon/menow_home_ass/notebooks/survival_analysis/survival_analysis_results.csv

Significance count by section:
  Phase 1A: 48 tests, 17 significant (raw), 12 significant (FDR)
  Phase 1B: 32 tests, 4 significant (raw), 3 significant (FDR)
  Phase 1C: 36 tests, 3 significant (raw), 0 significant (FDR)
  Phase 1D: 48 tests, 12 significant (raw), 4 significant (FDR)
  Phase 1E: 4 tests, 3 significant (raw), 3 significant (FDR)
